## **1. Environment Setup**

In [2]:
!pip install -q \
"transformers==5.0.0" \
 "peft==0.18.1" \
 "trl==1.1.0" \
 "accelerate==1.13.0" \
 "datasets==4.8.4" \
 "sentencepiece==0.2.1" \
 "protobuf==5.29.6"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 105.6 MB/s eta 0:00:00


In [9]:
import sys
import torch

print("python :",sys.version.split()[0])
print("pytorch :",torch.__version__)

print("GPU :",torch.cuda.is_available())
print("Total GPU",torch.cuda.device_count())
print("GPU Name :",torch.cuda.get_device_name(0))

props=torch.cuda.get_device_properties(0)
print(f"GPU Memory :{props.total_memory/1024**3:.2f} GB")

python : 3.13.15
pytorch : 2.11.0+cu128
GPU : True
Total GPU 1
GPU Name : Tesla T4
GPU Memory :14.56 GB


## **2. Load the base model in FP16**:
We load the model in half precision (`torch.float16`). On T4 this is the sweet spot — faster than FP32

In [15]:
import torch
from transformers import AutoTokenizer,AutoModelForCausalLM

model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer=AutoTokenizer.from_pretrained(model_id)

if tokenizer.pad_token is None:
  tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side='right'

model=AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="auto"
)

model.config.use_cache=False
model.config.pretraining_tp=1

n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_params/1e6:.1f} M")
print(f"Memory footprint: {model.get_memory_footprint()/1024**3:.2f} GB")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Total parameters: 1100.0 M
Memory footprint: 2.05 GB


## **3. Define a prompt format and test the BASE model**

We use TinyLlama's chat template. Task = text-to-SQL. Given a table schema and a question, produce a SQL query.

In [26]:
def build_prompt(schema: str, question: str) -> str:
    system = (
        "You are a SQL assistant. Given a table schema and a question, "
        "reply with ONLY the SQL query, nothing else."
    )
    user = f"Schema:\n{schema}\n\nQuestion: {question}"
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": user},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [27]:
# testing build_prompt function
test_prompt_1 = build_prompt(
    schema="CREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT);",
    question="List the names of employees in the Engineering department earning more than 100000."
)

test_prompt_1

'<|system|>\nYou are a SQL assistant. Given a table schema and a question, reply with ONLY the SQL query, nothing else.</s>\n<|user|>\nSchema:\nCREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT);\n\nQuestion: List the names of employees in the Engineering department earning more than 100000.</s>\n<|assistant|>\n'

In [31]:
@torch.no_grad
def generate(prompt: str, max_new_tokens: int = 120) -> str:
    inputs=tokenizer(prompt,return_tensors="pt").to(model.device)

    outputs=model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,                                    #slight similar to temperature for variational output
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
        )

    input_length=inputs["input_ids"].shape[1]
    output=tokenizer.decode(outputs[0][input_length:],skip_special_tokens=True)
    return output


output = generate(test_prompt_1)
print(output)

To answer this question, you can use the following SQL query:

```
SELECT name
FROM employees
WHERE department = 'Engineering'
AND salary > 100000;
```

This query will return a list of names of employees in the Engineering department who have a salary greater than 100,000.


In [35]:
# Three example prompts we will reuse AFTER fine-tuning for direct comparison.
examples = [
    {
        "schema":  "CREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT);",
        "question":"List the names of employees in the Engineering department earning more than 100000.",
    },
    {
        "schema":  "CREATE TABLE orders (order_id INT, customer_id INT, amount FLOAT, order_date DATE);",
        "question":"What is the total order amount per customer in 2024?",
    },
    {
        "schema":  "CREATE TABLE movies (title TEXT, year INT, rating FLOAT, genre TEXT);",
        "question":"Show the top 5 highest rated horror movies released after 2015.",
    },
]

print("BASE MODEL (before fine-tuning)")
print("-"*70)
base_outputs = []
for i, p in enumerate(examples, 1):
    prompt = build_prompt(p["schema"], p["question"])
    ans = generate(prompt)
    base_outputs.append(ans)
    print(f"\ne.g {i}")
    print("Q:", p["question"])
    print("A:", ans)

BASE MODEL (before fine-tuning)
----------------------------------------------------------------------

e.g 1
Q: List the names of employees in the Engineering department earning more than 100000.
A: To answer this question, you can use the following SQL query:

```
SELECT name
FROM employees
WHERE department = 'Engineering'
AND salary > 100000;
```

This query will return a list of names of employees in the Engineering department who have a salary greater than 100,000.

e.g 2
Q: What is the total order amount per customer in 2024?
A: To answer the question, you need to use the `SUM` function to calculate the total order amount per customer in 2024. Here's the SQL query:

```sql
SELECT SUM(amount) AS total_order_amount_per_customer
FROM orders
WHERE year(order_date) = 2024
GROUP BY customer_id;
```

This query will return a single row with the total order amount per customer in 2024.

e.g 3
Q: Show the top 5 highest rated horror movies released after 2015.
A: To show the top 5 highest 

## **4. Load and prepare the dataset**
We use `b-mc2/sql-create-context` — a compact text-to-SQL dataset with `(question, context, answer)` triples. We'll take a small slice so training finishes in a few minutes on T4.

In [43]:
from datasets import load_dataset

data = load_dataset("b-mc2/sql-create-context",split="train")
print("Full dataset size:", len(data))
print("Example row     :", data[0])

data=data.shuffle(seed=42).select(range(5000))
print("\nSmall dataset size:", len(data))

split=train_test_split=data.train_test_split(test_size=0.1,seed=42)
train_data=split["train"]
test_data=split["test"]
print("Train:", len(train_data), " test:", len(test_data))

Full dataset size: 78577
Example row     : {'answer': 'SELECT COUNT(*) FROM head WHERE age > 56', 'question': 'How many heads of the departments are older than 56 ?', 'context': 'CREATE TABLE head (age INTEGER)'}

Small dataset size: 5000
Train: 4500  test: 500


In [45]:
print("Schema:", train_data[0]["context"])
print("="*70)
print("Question:", train_data[0]["question"])
print("="*70)
print("Answer:", train_data[0]["answer"])
print("="*70)

Schema: CREATE TABLE table_name_12 (car__number INTEGER, points VARCHAR)
Question: What is the average car number of all the drivers with 109 points?
Answer: SELECT AVG(car__number) FROM table_name_12 WHERE points = "109"


In [47]:
def format_example(row):

    system=(
        'You are a SQL assistant. Given a table schema and a question, '
        'reply with ONLY the SQL query, nothing else.'
    )
    user=f"Schema:\n{row['context']}\n\nQuestion: {row['question']}"
    assistant=row['answer']

    message=[
        {"role":"system","content":system},
        {"role":"user","content":user},
        {"role":"assistant","content":assistant},
    ]

    text=tokenizer.apply_chat_template(message,tokenize=False,add_generation_prompt=True)
    return {"text":text}

train_data=train_data.map(format_example)
test_data=test_data.map(format_example)

print("\n--- Formatted training example ---\n")
print(train_data[0]["text"][:800])# underatand the dataset
train_data[0].keys()



Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]


--- Formatted training example ---

<|system|>
You are a SQL assistant. Given a table schema and a question, reply with ONLY the SQL query, nothing else.</s>
<|user|>
Schema:
CREATE TABLE table_name_12 (car__number INTEGER, points VARCHAR)

Question: What is the average car number of all the drivers with 109 points?</s>
<|assistant|>
SELECT AVG(car__number) FROM table_name_12 WHERE points = "109"</s>
<|assistant|>



dict_keys(['answer', 'question', 'context', 'text'])

## **5. Attach LoRA Adapters**
LoRA inserts low-rank trainable matrices into specific linear layers (attention projections) while freezing the base weights. For Llama-family models the common target modules are `q_proj`, `k_proj`, `v_proj`, `o_proj` (and optionally the MLP projections).

In [48]:
from peft import LoraConfig,get_peft_model

# Enable gradient checkpointing to save VRAM during backward pass
model.gradient_checkpointing_enable()
model.enable_input_require_grads()     # needed because base params are frozen

lora_config=LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"]
)

model=get_peft_model(model,lora_config)
model.print_trainable_parameters()

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


## **6. Train**
We use `SFTTrainer` from TRL.

Settings chosen for T4 (16 GB):
- `per_device_train_batch_size=2`, `gradient_accumulation_steps=8` → effective batch 16
- `max_seq_length=512`
- `fp16=True` (T4 supports FP16, not BF16)
- 1 epoch over 3000 examples ≈ ~5–8 minutes


In [53]:
from trl import SFTTrainer,SFTConfig

output_dir="./results"

sft_config = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    optim="adamw_torch",
    fp16=True,
    bf16=False,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="epoch",
    report_to="none",
)

def tokenize(batch):
    out = tokenizer(
        batch["text"],
        truncation=True,
        max_length=512,
        padding=False,
    )
    return out

train_tok = train_data.map(tokenize, batched=True, remove_columns=["text"])
test_tok = test_data.map (tokenize, batched=True, remove_columns=["text"])

In [58]:
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_tok,
    eval_dataset=test_tok,
)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss
50,0.592773,0.585661
100,0.560840,0.546502
150,0.506449,0.529960
200,0.488926,0.521475
250,0.523818,0.518068


TrainOutput(global_step=282, training_loss=0.6144233707417833, metrics={'train_runtime': 835.7124, 'train_samples_per_second': 5.385, 'train_steps_per_second': 0.337, 'total_flos': 4092417538228224.0, 'train_loss': 0.6144233707417833})

In [59]:
# Re-enable cache for fast inference
model.config.use_cache = True
model.eval()

print("FINE-TUNED MODEL (after LoRA)")
print("="*70)
for i, p in enumerate(examples, 1):
    prompt = build_prompt(p["schema"], p["question"])
    ans = generate(prompt)
    print(f"\ne.g {i} ")
    print("Q:     ", p["question"])
    print("BEFORE:", base_outputs[i-1])
    print("="*70)
    print("AFTER :", ans)
    print("="*70)


FINE-TUNED MODEL (after LoRA)

e.g 1 
Q:      List the names of employees in the Engineering department earning more than 100000.
BEFORE: To answer this question, you can use the following SQL query:

```
SELECT name
FROM employees
WHERE department = 'Engineering'
AND salary > 100000;
```

This query will return a list of names of employees in the Engineering department who have a salary greater than 100,000.
AFTER : SELECT name FROM employees WHERE department = "Engineering" AND salary > 100000

e.g 2 
Q:      What is the total order amount per customer in 2024?
BEFORE: To answer the question, you need to use the `SUM` function to calculate the total order amount per customer in 2024. Here's the SQL query:

```sql
SELECT SUM(amount) AS total_order_amount_per_customer
FROM orders
WHERE year(order_date) = 2024
GROUP BY customer_id;
```

This query will return a single row with the total order amount per customer in 2024.
AFTER : SELECT SUM(amount) FROM orders WHERE customer_id = 2024

e

## **8. Save the Adapters**
LoRA adapters are tiny (a few MB). You save *only* the adapter, not the whole base model.

In [60]:
ADAPTER_DIR = "./tinyllama-sql-lora-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

import os
total = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
    if os.path.isfile(os.path.join(ADAPTER_DIR, f))
)
print(f"Adapter size on disk: {total/1024**2:.2f} MB")

Adapter size on disk: 20.67 MB


## **Loading the adapter later (reference code)**

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    dtype=torch.float16,
    device_map="auto",
)
model = PeftModel.from_pretrained(base, "./tinyllama-sql-lora-adapter")
tokenizer = AutoTokenizer.from_pretrained("./tinyllama-sql-lora-adapter")